# 04 — Komparasi Model & Penyimpanan Artefak (Model Comparison)
**NapasLoka: Hyperparameter Tuning, Pelatihan XGBoost vs LightGBM, dan Persistensi Model**

Notebook ini merepresentasikan inti dari eksperimen pemodelan Machine Learning di Bab 4:
1. **Hyperparameter Tuning** dengan `TimeSeriesSplit(n_splits=3)` pada data latih.
2. **Pelatihan Model** XGBoost Regressor dan LightGBM Regressor.
3. **Pengujian Kinerja Komparatif** pada data uji yang belum pernah dilihat (*unseen data*).
4. **Penyimpanan Artefak Lengkap** ke direktori `artifacts/experiments/` sesuai standar sistem NapasLoka.


---
### 1. Setup Lingkungan & Import Modul


In [ ]:
import os, sys, json, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('napasloka'):
        !git clone https://github.com/Nikanzz/napasloka.git
    %cd napasloka
    !pip install -q xgboost lightgbm shap pyyaml joblib matplotlib scikit-learn
    ROOT_DIR = Path("/content/napasloka")
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
        ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features.lag_features import create_lag_features
from src.features.scenarios import resolve_scenario
from src.data.preprocessing import chronological_train_test_split
from src.modeling.training import create_estimator
from src.modeling.tuning import tune_on_training_set
from src.evaluation.metrics import calculate_regression_metrics
from src.interpretation.shap_analysis import calculate_tree_shap, mean_absolute_shap
from src.interpretation.feature_importance import gain_feature_importance
from src.utils.config import load_yaml_config

print("✓ Modul modeling and comparison berhasil dimuat.")


---
### 2. Parameter Eksperimen & Pembentukan Data


In [ ]:
STATION_ID = "pantai-makmur"
POLLUTANT = "co2"
SCENARIO = "scenario_2"
SAVE_ARTIFACTS = True

# Pemuatan data stasiun
df_raw = pd.read_csv(ROOT_DIR / "data" / "processed" / f"jabodetabek_{POLLUTANT}_dataset.csv")
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")

coord_id = "-6.1_107.0"
df_station = df_raw[(df_raw["location"] == coord_id) | (df_raw["location"] == STATION_ID)].sort_values("datetime").reset_index(drop=True)
df_feat = create_lag_features(df_station, pollutant=POLLUTANT, scenario=SCENARIO, drop_incomplete_rows=True)

scenario_def = resolve_scenario(POLLUTANT, SCENARIO)
feature_cols = list(scenario_def.feature_names)
target_col = scenario_def.pollutant_column

train_df, test_df = chronological_train_test_split(df_feat, test_fraction=0.2)
x_train, y_train = train_df[feature_cols], train_df[target_col]
x_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Data Latih: {len(x_train):,} sampel | Data Uji: {len(x_test):,} sampel | Fitur: {len(feature_cols)} kolom")


---
### 3. Hyperparameter Tuning pada Data Latih


In [ ]:
models = ["xgboost", "lightgbm"]
tuned_estimators = {}
best_params = {}

for algo in models:
    print(f"Tuning {algo.upper()} dengan TimeSeriesSplit(3)...")
    base_est = create_estimator(algo, parameters={"random_state": 42})
    search = tune_on_training_set(base_est, algo, x_train, y_train)
    tuned_estimators[algo] = search.best_estimator_
    best_params[algo] = search.best_params_
    print(f"✓ Best params {algo.upper()}: {search.best_params_}")


---
### 4. Pelatihan Model & Evaluasi Kinerja (Data Uji)


In [ ]:
results = {}

for algo in models:
    model = tuned_estimators[algo]
    y_pred = model.predict(x_test)
    metrics = calculate_regression_metrics(y_test, y_pred)
    results[algo] = {"model": model, "y_pred": y_pred, "metrics": metrics}

# Tabel Perbandingan Kinerja
df_comp = pd.DataFrame({
    algo.upper(): {
        "MAE (ppm)": results[algo]["metrics"]["mae"],
        "RMSE (ppm)": results[algo]["metrics"]["rmse"],
        "MAPE (%)": results[algo]["metrics"]["mape"],
        "R² Score": results[algo]["metrics"]["r2"],
    }
    for algo in models
}).T

print("=== HASIL KOMPARASI METRIK EVALUASI (DATA UJI) ===")
display(df_comp)


---
### 5. Simpan Model & Artefak ke `artifacts/experiments/`


In [ ]:
if SAVE_ARTIFACTS:
    for algo in models:
        artifact_dir = ROOT_DIR / "artifacts" / "experiments" / STATION_ID / POLLUTANT / SCENARIO / algo
        artifact_dir.mkdir(parents=True, exist_ok=True)
        
        # 1. Simpan Model (.joblib)
        joblib.dump(results[algo]["model"], artifact_dir / "model.joblib")
        # 2. Simpan Best Params (.json)
        with (artifact_dir / "best_params.json").open("w", encoding="utf-8") as f:
            json.dump(best_params[algo], f, indent=2)
        # 3. Simpan Metrics (.json)
        with (artifact_dir / "metrics.json").open("w", encoding="utf-8") as f:
            json.dump(results[algo]["metrics"], f, indent=2)
        # 4. Simpan Prediksi Data Uji (predictions.csv)
        pd.DataFrame({
            "datetime": test_df["datetime"].astype(str),
            "actual": y_test.values,
            "predicted": results[algo]["y_pred"],
            "residual": y_test.values - results[algo]["y_pred"],
        }).to_csv(artifact_dir / "predictions.csv", index=False)
        
        print(f"✓ Berhasil menyimpan artefak {algo.upper()} ke: {artifact_dir}")
